In [6]:
from langgraph.graph import StateGraph,END,START
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from typing import TypedDict
import os
from langchain_groq import ChatGroq

In [ ]:
load_dotenv(dotenv_path=".env", override=True)

True

In [4]:
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [5]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [10]:
class jokestate(TypedDict):
    title:str
    jokes:str
    explaination:str


In [9]:
graph=StateGraph(jokestate)

In [11]:
def joke(state:jokestate)->jokestate:
    prompt=f"Generate the joke of the title{state['title']} in a funny way."
    response=model.invoke(prompt).content
    return {'jokes':response}

In [12]:
def explaination(state:jokestate)->jokestate:
    prompt=f"Explain the joke {state['jokes']}of the title{state['title']} in a funny way."
    response=model.invoke(prompt).content
    return {'explaination':response}

In [13]:
graph.add_node('joke',joke)
graph.add_node('explaination',explaination)

In [14]:
graph.add_edge(START,'joke')
graph.add_edge('joke','explaination')
graph.add_edge('explaination',END)
checkpoint=InMemorySaver()
workflow=graph.compile(checkpointer=checkpoint)

In [15]:
config1={'configurable':{'thread_id':'1'}}
workflow.invoke({'title':'samosa'},config=config1)

{'title': 'samosa',
 'explaination': 'A delicious joke. Let\'s break it down:\n\nThe joke is a play on words, using the characteristics of a samosa (a type of fried or baked pastry) to create a humorous analogy about emotional struggles.\n\nHere\'s how it works:\n\n1. **"Crunchy" on the outside**: Samosas are known for their crispy exterior. In this joke, "crunchy" is used to describe the samosa\'s emotional state, implying that it\'s tough or hardened on the outside, possibly as a defense mechanism.\n2. **"Empty" on the inside**: Samosas are typically filled with a savory mixture of ingredients. The joke says the samosa is "empty" on the inside, which is a clever way to describe feelings of hollowness, sadness, or emotional emptiness.\n3. **"Wrap its head around its problems"**: This phrase is a common idiom that means to understand or come to terms with one\'s issues. The joke uses it literally, referencing the fact that a samosa is a wrapped food item.\n4. **"Fill the void"**: Anoth